# # 120 — Hướng A Screening: CMP-exp (fix κ) & ASF-Lite (15 epoch, seed 0)
#
# **Mục tiêu:** kiểm chứng 2 cải tiến kiến trúc trực tiếp giải quyết failure modes đã chẩn đoán:
# 1. **CMP-exp (`cmp_scale_mode="exp"`):** sửa lỗi quên `.exp()` ở `logit_scale` (`κ = exp(θ) ∈ [10, 64]`)
#    + `hardmax` + `k-means init`, xóa bỏ trần softmax 13.32% dưới KD.
# 2. **ASF-Lite (`model_key="asf_lite"`):** Haar DWT trên `layer2` (32×32) & `layer3` (16×16),
#    dùng GAP(LL) + RMS(LH, HL, HH) không triệt tiêu dấu, bỏ 8 nhánh Conv2d+BN lặp để giữ latency ~ResNet18.
#
# **Controls seed 0 (cùng split/protocol):** B1 ResNet18-CE `0.9483` (060a), ResNet18-KD `0.9550` (110b).
# **Strict protocol:** KHÔNG chạm tập `test` trong screening; chỉ kiểm trên `val` + đo latency batch-1.

In [ ]:
# ---- CONFIG ----
import os

RUN_ID = "120-h1a-screen-v1"
SEED = 0
EPOCHS = 15
KD_ALPHA, KD_T = 0.5, 4.0
EXPERIMENTS = (
    ("cmp_exp_ce", dict(model_key="cmp", cmp_n_proto=2, cmp_agg="hardmax", cmp_kmeans_init=True,
                        cmp_scale_mode="exp", kd_alpha=0.0)),
    ("cmp_exp_kd", dict(model_key="cmp", cmp_n_proto=2, cmp_agg="hardmax", cmp_kmeans_init=True,
                        cmp_scale_mode="exp", kd_alpha=KD_ALPHA, kd_T=KD_T)),
    ("asf_lite_ce", dict(model_key="asf_lite", asf_out_init="small", asf_gate_ent=0.05, kd_alpha=0.0)),
    ("asf_lite_kd", dict(model_key="asf_lite", asf_out_init="small", asf_gate_ent=0.05,
                         kd_alpha=KD_ALPHA, kd_T=KD_T)),
)
REF_S0 = {"ce": 0.9483, "kd": 0.9550}
TEACHER_SEED = 2
SPLIT_SHA_EXPECTED = "cc70020a85091b4d86fd519e6246b2ca397a20bca551803d532bf8cc1254645a"
SMOKE = os.environ.get("PILOT_SMOKE", "0" if os.path.isdir("/kaggle/input") else "1") == "1"
if SMOKE:
    EPOCHS = 1

In [ ]:
# ---- ENV + manifest + teacher verify ----
import glob
import hashlib
import json
import sys

import pandas as pd
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0 if sys.platform == "darwin" else 4
print(f"run_id: {RUN_ID} | smoke: {SMOKE} | device: {device}")


def find_single(pattern_list: list[str], desc: str) -> str:
    for pat in pattern_list:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return hits[0]
    raise AssertionError(f"FAIL-FAST: khong tim thay {desc}: {pattern_list}")


manifest_path = find_single(
    ["/kaggle/input/**/split_manifest.csv",
     "../../notebooks/010-data-audit/split_manifest.smoke.csv",
     "split_manifest.smoke.csv"], "split manifest")
manifest = pd.read_csv(manifest_path)
manifest = manifest[manifest.split != "test"].reset_index(drop=True)
audit_path = find_single(
    ["/kaggle/input/**/audit_report.json",
     "../../notebooks/010-data-audit/audit_report.smoke.json"], "audit report")
with open(audit_path) as f:
    audit = json.load(f)
if not SMOKE:
    assert audit["split_sha256"] == SPLIT_SHA_EXPECTED, "FAIL-FAST: split lệch chuẩn"
print("manifest:", manifest_path, "| rows:", len(manifest))

DATA_ROOT = None
for hit in sorted(glob.glob("/kaggle/input/**/NWPU-RESISC45", recursive=True)) + [
        "../../data/raw/punfake---resisc45/NWPU-RESISC45"]:
    if os.path.isdir(hit) and len([d for d in os.listdir(hit) if os.path.isdir(os.path.join(hit, d))]) >= 40:
        DATA_ROOT = hit
        break
assert DATA_ROOT, "FAIL-FAST: dataset root"

In [ ]:
# ===== embedded from src/models.py (single source of logic) =====
# """Nguon logic dung chung (single source) cho moi notebook: build model, train, eval.
# Được nbmake.py embed vào notebook khi build — notebook tự chứa bản copy sinh tự động,
# KHÔNG sửa trực tiếp trong notebook; sửa file này rồi rebuild.
# """
from __future__ import annotations

import copy
import itertools
import json
import math
import os
import random
import time
from collections.abc import Callable
from dataclasses import asdict, dataclass, field

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

# ---------------------------------------------------------------- utils

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def count_params(model: nn.Module) -> dict:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable}


# ---------------------------------------------------------------- data


@dataclass
class TrainConfig:
    model_key: str = "resnet18"          # smallcnn | resnet18 | mobilenetv3s | efficientnetb0 | convnextt | asf | cmp
    num_classes: int = 45
    seed: int = 0
    epochs: int = 15
    batch_size: int = 128
    lr_backbone: float = 1e-4
    lr_head: float = 1e-3
    weight_decay: float = 1e-4
    warmup_frac: float = 0.05
    early_stop_patience: int = 5
    amp: bool = True
    num_workers: int = 4
    input_size: int = 256
    # ASF
    asf_fixed_fusion: bool = False        # A1 (True) vs A2 (False)
    asf_feature_dim: int = 128
    asf_detail_mode: str = "haar"         # "haar" | "conv" (đối chứng spatial)
    asf_out_init: str = "zero"            # "zero" | "small" (std=1e-3 — cho gradient gate sống từ đầu)
    asf_gate_ent: float = 0.0             # >0: entropy regularizer chống gate sụp 1-hot
    asf_gate_ent_anneal: int = 0          # >0: λ entropy giảm tuyến tính về 0 sau N epoch (ASF-EA)
    asf_gate_descriptor: str = "gap"      # "gap" | "energy" (ASF-E: [RMS, mean|·|] 16-dim → BN → Linear)
    cmp_kmeans_init: bool = False         # khởi tạo prototype bằng k-means trên embedding
    cmp_agg: str = "lse"                  # "lse" (v1) | "hardmax" (CMP-P: chỉ winner nhận gradient)
    cmp_pair_loss: bool = False           # CMP-PG: pair loss chiều rõ trên confusion graph
    cmp_pair_margin: float = 0.1
    cmp_pair_weight: float = 0.5
    cmp_pair_warmup: int = 3              # λ = weight · min(1, (epoch+1)/warmup)
    kd_alpha: float = 0.0                 # >0: KD bật — L = (1−α)·CE + α·T²·KL(teacher‖student)
    kd_T: float = 4.0                     # temperature distillation
    # CMP
    cmp_n_proto: int = 1                  # C1/C2: 1 hoặc 2, 4
    cmp_confusion: bool = False           # C3
    cmp_conf_margin: float = 0.3
    cmp_conf_weight: float = 0.5
    cmp_diversity: bool = False           # C4
    cmp_div_weight: float = 0.01
    cmp_embed_dim: int = 512
    cmp_scale_mode: str = "linear"        # "linear" (v1/v2/v3) | "exp" (sửa trần softmax κ = exp(θ))
    # chung
    label_smoothing: float = 0.0
    extra: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return asdict(self)


class ResiscDataset(Dataset):
    """Đọc ảnh theo manifest (image_id relpath từ root NWPU-RESISC45)."""

    def __init__(self, manifest: pd.DataFrame, root: str, train: bool, input_size: int = 256):
        self.items = manifest.reset_index(drop=True)
        self.root = root
        self.train = train
        self.input_size = input_size

    def __len__(self) -> int:
        return len(self.items)

    def _path(self, i: int) -> str:
        r = self.items.iloc[i]
        return os.path.join(self.root, r.image_id)

    def __getitem__(self, i: int):
        img = Image.open(self._path(i)).convert("RGB")
        img = train_augment(img, self.input_size) if self.train else eval_augment(img, self.input_size)
        return img, int(self.items.iloc[i].class_index)


def train_augment(img: Image.Image, size: int) -> torch.Tensor:
    """Augmentation chung (thuần PIL, tránh phụ thuộc torchvision.transforms)."""
    import torchvision.transforms.functional as TF

    # RandomResizedCrop(scale 0.7–1.0) + hflip + vflip
    w, h = img.size
    scale = random.uniform(0.7, 1.0)
    ratio = random.uniform(3 / 4, 4 / 3)
    area = w * h * scale
    cw = min(w, round(math.sqrt(area * ratio)))
    ch = min(h, round(math.sqrt(area / ratio)))
    x = random.randint(0, w - cw)
    y = random.randint(0, h - ch)
    img = img.crop((x, y, x + cw, y + ch)).resize((size, size), Image.BILINEAR)
    if random.random() < 0.5:
        img = TF.hflip(img)
    if random.random() < 0.5:
        img = TF.vflip(img)
    return TF.normalize(TF.to_tensor(img), IMAGENET_MEAN, IMAGENET_STD)


def eval_augment(img: Image.Image, size: int) -> torch.Tensor:
    import torchvision.transforms.functional as TF

    img = img.resize((size, size), Image.BILINEAR)
    return TF.normalize(TF.to_tensor(img), IMAGENET_MEAN, IMAGENET_STD)


def make_loaders(manifest: pd.DataFrame, root: str, cfg: TrainConfig):
    tr = manifest[manifest.split == "train"]
    va = manifest[manifest.split == "val"]
    dl_train = DataLoader(ResiscDataset(tr, root, True, cfg.input_size), batch_size=cfg.batch_size,
                          shuffle=True, num_workers=cfg.num_workers, pin_memory=True, drop_last=False)
    dl_val = DataLoader(ResiscDataset(va, root, False, cfg.input_size), batch_size=cfg.batch_size,
                        shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    return dl_train, dl_val


# ---------------------------------------------------------------- model registry & factory

ModelBuilder = Callable[[TrainConfig, bool], nn.Module]
MODEL_REGISTRY: dict[str, ModelBuilder] = {}


def register_model(key: str) -> Callable[[ModelBuilder], ModelBuilder]:
    """Decorator đăng ký model builder vào MODEL_REGISTRY theo model_key."""
    def _decorator(fn: ModelBuilder) -> ModelBuilder:
        MODEL_REGISTRY[key] = fn
        return fn
    return _decorator


def _build_torchvision_classifier(
    factory_name: str,
    weights_name: str,
    head_setter: Callable[[nn.Module, int], None],
    num_classes: int,
    pretrained: bool,
) -> nn.Module:
    """Factory dùng chung cho các torchvision backbone (tránh lặp code weights/head)."""
    import torchvision

    weights = getattr(torchvision.models, weights_name).IMAGENET1K_V1 if pretrained else None
    model = getattr(torchvision.models, factory_name)(weights=weights)
    head_setter(model, num_classes)
    return model


def _set_fc(m: nn.Module, nc: int) -> None:
    m.fc = nn.Linear(m.fc.in_features, nc)


def _set_classifier_idx(idx: int) -> Callable[[nn.Module, int], None]:
    def _setter(m: nn.Module, nc: int) -> None:
        m.classifier[idx] = nn.Linear(m.classifier[idx].in_features, nc)
    return _setter


_TV_BACKBONE_SPECS: dict[str, tuple[str, str, Callable[[nn.Module, int], None]]] = {
    "resnet18": ("resnet18", "ResNet18_Weights", _set_fc),
    "mobilenetv3s": ("mobilenet_v3_small", "MobileNet_V3_Small_Weights", _set_classifier_idx(3)),
    "efficientnetb0": ("efficientnet_b0", "EfficientNet_B0_Weights", _set_classifier_idx(1)),
    "convnextt": ("convnext_tiny", "ConvNeXt_Tiny_Weights", _set_classifier_idx(2)),
}

for _tv_key, (_fn, _wn, _setter) in _TV_BACKBONE_SPECS.items():
    def _make_tv_builder(fn_name: str = _fn, w_name: str = _wn, setter: Callable = _setter) -> ModelBuilder:
        return lambda cfg, pretrained: _build_torchvision_classifier(
            fn_name, w_name, setter, cfg.num_classes, pretrained
        )
    MODEL_REGISTRY[_tv_key] = _make_tv_builder()


@register_model("smallcnn")
def _build_smallcnn(cfg: TrainConfig, pretrained: bool) -> nn.Module:
    return SmallCNN(cfg.num_classes)


@register_model("asf")
def _build_asf(cfg: TrainConfig, pretrained: bool) -> nn.Module:
    return ASFResNet18(
        cfg.num_classes,
        pretrained=pretrained,
        fixed_fusion=cfg.asf_fixed_fusion,
        feature_dim=cfg.asf_feature_dim,
        detail_mode=cfg.asf_detail_mode,
        out_init=cfg.asf_out_init,
        gate_descriptor=cfg.asf_gate_descriptor,
    )


@register_model("asf_lite")
def _build_asf_lite(cfg: TrainConfig, pretrained: bool) -> nn.Module:
    return ASFLiteResNet18(
        cfg.num_classes,
        pretrained=pretrained,
        feature_dim=cfg.asf_feature_dim,
        out_init=cfg.asf_out_init,
    )


@register_model("cmp")
def _build_cmp(cfg: TrainConfig, pretrained: bool) -> nn.Module:
    return CMPResNet18(
        cfg.num_classes,
        pretrained=pretrained,
        n_proto=cfg.cmp_n_proto,
        embed_dim=cfg.cmp_embed_dim,
        confusion=cfg.cmp_confusion,
        conf_margin=cfg.cmp_conf_margin,
        diversity=cfg.cmp_diversity,
        agg=cfg.cmp_agg,
        pair=cfg.cmp_pair_loss,
        pair_margin=cfg.cmp_pair_margin,
        scale_mode=cfg.cmp_scale_mode,
    )


def build_model(cfg: TrainConfig, pretrained: bool = True) -> nn.Module:
    """B0–B4 và 2 proposed qua MODEL_REGISTRY. pretrained=False chỉ dùng cho smoke/offline load."""
    builder = MODEL_REGISTRY.get(cfg.model_key)
    if builder is None:
        raise ValueError(f"unknown model_key: {cfg.model_key} (available: {sorted(MODEL_REGISTRY)})")
    return builder(cfg, pretrained)


class SmallCNN(nn.Module):
    """B0 — CNN nhỏ train từ đầu, cấu trúc hợp lý (3 block, BN, GAP)."""

    def __init__(self, num_classes: int = 45):
        super().__init__()
        ch = [3, 32, 64, 128]
        layers = []
        for a, b in itertools.pairwise(ch):
            layers += [nn.Conv2d(a, b, 3, padding=1, bias=False), nn.BatchNorm2d(b), nn.ReLU(inplace=True),
                       nn.Conv2d(b, b, 3, padding=1, bias=False), nn.BatchNorm2d(b), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128, num_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(F.adaptive_avg_pool2d(self.features(x), 1))


# ---------------------------------------------------------------- ASF-ResNet18


def haar_dwt2(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Haar DWT 2D level-1 trên tensor (B,C,H,W), H/W chẵn (crop trước nếu lẻ).
    Trả về (LL, LH, HL, HH) — mỗi cái (B,C,H/2,W/2)."""
    if x.shape[-2] % 2 or x.shape[-1] % 2:
        x = x[..., : x.shape[-2] // 2 * 2, : x.shape[-1] // 2 * 2]
    a = x[..., 0::2, 0::2]
    b = x[..., 0::2, 1::2]
    c = x[..., 1::2, 0::2]
    d = x[..., 1::2, 1::2]
    ll = (a + b + c + d) / 4.0
    lh = (a + b - c - d) / 4.0
    hl = (a - b + c - d) / 4.0
    hh = (a - b - c + d) / 4.0
    return ll, lh, hl, hh


def _energy_gate_features(raw_comps: list[torch.Tensor]) -> torch.Tensor:
    """Descriptor năng lượng thô [RMS, mean|·|] trên mỗi subband — 16-dim tổng."""
    rms, mar = [], []
    for c in raw_comps:
        rms.append(c.pow(2).mean(dim=(1, 2, 3)).add(1e-6).sqrt())
        mar.append(c.abs().mean(dim=(1, 2, 3)))
    return torch.stack(rms + mar, dim=1)


class ASFResNet18(nn.Module):
    """Adaptive Scale–Frequency ResNet18.

    - Giữ backbone ResNet18 pretrained NGUYÊN VẸN (không thay conv nào).
    - Lấy feature map layer2 (128ch, 32×32) và layer4 (512ch, 8×8) → Haar DWT → 8 thành phần.
    - Mỗi thành phần: 1×1 conv → F dims → BN+ReLU → GAP → vector f_i.
    - Gate (theo ảnh): MLP(8F→8) softmax; hoặc fixed_fusion=True: softmax over learnable logits (A1).
    - Fused = Σ w_i f_i → Linear (F→512) → residual cộng vào GAP(layer4) → head.
    - Giữ nguyên tên attribute (`proj`, `gate`, `gate_en`, `out_proj`, `head`, `detail_conv`, `fixed_logits`)
      để tương thích state_dict các checkpoint đã train.
    """

    def __init__(self, num_classes: int = 45, pretrained: bool = True, fixed_fusion: bool = False,
                 feature_dim: int = 128, detail_mode: str = "haar", out_init: str = "zero",
                 gate_descriptor: str = "gap"):
        super().__init__()
        import torchvision

        bb = torchvision.models.resnet18(
            weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        )
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool)
        self.layer1, self.layer2, self.layer3, self.layer4 = bb.layer1, bb.layer2, bb.layer3, bb.layer4
        self.fixed_fusion = fixed_fusion
        self.feature_dim = feature_dim
        self.detail_mode = detail_mode          # "haar" (A1/A2) | "conv" (đối chứng spatial cùng chi phí)
        self.gate_descriptor = gate_descriptor  # "gap" | "energy" (ASF-E)
        self.n_components = 8                   # 2 stage × (ORIG + 3 detail)
        F_ = feature_dim
        self.proj = nn.ModuleList([
            nn.Sequential(nn.Conv2d(c, F_, 1, bias=False), nn.BatchNorm2d(F_), nn.ReLU(inplace=True))
            for c in (128, 128, 128, 128, 512, 512, 512, 512)
        ])
        self.pool = nn.AdaptiveAvgPool2d(1)
        if detail_mode == "conv":
            self.detail_conv = nn.ModuleList([
                nn.Conv2d(c, c, 3, stride=2, padding=1, groups=c, bias=False)
                for c in (128, 128, 128, 512, 512, 512)
            ])
        if fixed_fusion:
            self.fixed_logits = nn.Parameter(torch.zeros(self.n_components))
        elif gate_descriptor == "energy":
            self.gate_en = nn.Sequential(
                nn.BatchNorm1d(2 * self.n_components),
                nn.Linear(2 * self.n_components, self.n_components),
            )
        else:
            self.gate = nn.Linear(F_ * self.n_components, self.n_components)
        self.out_proj = nn.Linear(F_, 512)
        if out_init == "small":
            nn.init.normal_(self.out_proj.weight, std=1e-3)
        else:
            nn.init.zeros_(self.out_proj.weight)
        nn.init.zeros_(self.out_proj.bias)
        self.head = nn.Linear(512, num_classes)
        self._last_gates: torch.Tensor | None = None

    def _raw_components(self, f2: torch.Tensor, f4: torch.Tensor) -> list[torch.Tensor]:
        """8 tensor thô: ORIG≡full feature map + 3 detail (haar hoặc conv học được)."""
        detail_extractors: dict[str, Callable[[int, torch.Tensor], list[torch.Tensor]]] = {
            "conv": lambda s_idx, fm: [self.detail_conv[s_idx * 3 + d](fm) for d in range(3)],
            "haar": lambda _s_idx, fm: list(haar_dwt2(fm)[1:]),
        }
        extractor = detail_extractors.get(self.detail_mode, detail_extractors["haar"])
        comps: list[torch.Tensor] = []
        for stage_idx, fm in enumerate((f2, f4)):
            comps.append(fm)
            comps.extend(extractor(stage_idx, fm))
        return comps

    def _backbone_feats(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h1 = self.layer1(self.stem(x))
        f2 = self.layer2(h1)                # 128ch, 32×32
        f4 = self.layer4(self.layer3(f2))   # 512ch, 8×8
        return f2, f4

    def _encode_components(self, raw: list[torch.Tensor]) -> torch.Tensor:
        """Chiếu 8 thành phần qua 1×1 conv → BN → ReLU → GAP → stack (B, 8, F)."""
        return torch.stack([self.pool(self.proj[i](c)).flatten(1) for i, c in enumerate(raw)], dim=1)

    def _gate_input(self, raw_comps: list[torch.Tensor], stack: torch.Tensor) -> torch.Tensor:
        if self.gate_descriptor == "energy":
            return self.gate_en(_energy_gate_features(raw_comps))
        return self.gate(stack.flatten(1))

    def _gate_weights_from_feats(self, raw: list[torch.Tensor], stack: torch.Tensor) -> torch.Tensor:
        if self.fixed_fusion:
            return torch.softmax(self.fixed_logits, dim=0).view(1, -1).expand(stack.size(0), -1)
        return torch.softmax(self._gate_input(raw, stack), dim=-1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f2, f4 = self._backbone_feats(x)
        raw = self._raw_components(f2, f4)
        stack = self._encode_components(raw)                        # (B, 8, F)
        w = self._gate_weights_from_feats(raw, stack)               # (B, 8)
        self._last_gates = None if self.fixed_fusion else w         # giữ gradient cho entropy regularizer
        fused = (stack * w.unsqueeze(-1)).sum(dim=1)                # (B, F)
        backbone_vec = self.pool(f4).flatten(1)                     # (B, 512)
        return self.head(backbone_vec + self.out_proj(fused))

    def gate_entropy_loss(self) -> torch.Tensor:
        """Phạt gate sụp 1-hot: (log K − H(w)) / log K ∈ [0,1], 0 khi đều."""
        if self.fixed_fusion or self._last_gates is None:
            return torch.zeros((), device=next(self.parameters()).device)
        p = self._last_gates.clamp_min(1e-8)
        h = -(p * p.log()).sum(-1).mean()
        return (math.log(self.n_components) - h) / math.log(self.n_components)

    def gate_weights(self, x: torch.Tensor) -> torch.Tensor:
        """Trả về (B, 8) để phân tích phân bố gate (chỉ dùng khi fixed_fusion=False)."""
        assert not self.fixed_fusion
        was_training = self.training
        self.eval()
        with torch.no_grad():
            f2, f4 = self._backbone_feats(x)
            raw = self._raw_components(f2, f4)
            stack = self._encode_components(raw)
            w = self._gate_weights_from_feats(raw, stack)
        if was_training:
            self.train()
        return w


class ASFLiteResNet18(nn.Module):
    """ASF-Lite (v3+): Haar DWT trên layer2 (32×32) & layer3 (16×16) không dùng conv 2D phụ.

    - Dùng GAP cho thành phần LL và RMS (non-cancelling energy) cho 3 subband cao tần (LH, HL, HH).
    - Band gate nhẹ (16→8) chọn lọc 8 dải tần-tỉ lệ; entropy regularizer dùng chung API `gate_entropy_loss`.
    - Loại bỏ 8 nhánh 1×1 Conv+BN lặp → giữ latency gần bằng ResNet18 gốc và giảm tham số phụ.
    """

    def __init__(self, num_classes: int = 45, pretrained: bool = True, feature_dim: int = 128,
                 out_init: str = "small"):
        super().__init__()
        import torchvision

        bb = torchvision.models.resnet18(
            weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        )
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool)
        self.layer1, self.layer2, self.layer3, self.layer4 = bb.layer1, bb.layer2, bb.layer3, bb.layer4
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.n_components = 8
        in_dim = 4 * (128 + 256)  # 4 subband × (layer2 128ch + layer3 256ch) = 1536
        self.gate = nn.Sequential(nn.LayerNorm(2 * self.n_components), nn.Linear(2 * self.n_components, 8))
        self.proj = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.out_proj = nn.Linear(feature_dim, 512)
        if out_init == "small":
            nn.init.normal_(self.out_proj.weight, std=1e-3)
        else:
            nn.init.zeros_(self.out_proj.weight)
        nn.init.zeros_(self.out_proj.bias)
        self.head = nn.Linear(512, num_classes)
        self._last_gates: torch.Tensor | None = None

    @staticmethod
    def _band_vectors(fm: torch.Tensor) -> list[torch.Tensor]:
        ll, lh, hl, hh = haar_dwt2(fm)
        return [
            ll.mean(dim=(2, 3)),
            lh.pow(2).mean(dim=(2, 3)).add(1e-6).sqrt(),
            hl.pow(2).mean(dim=(2, 3)).add(1e-6).sqrt(),
            hh.pow(2).mean(dim=(2, 3)).add(1e-6).sqrt(),
        ]

    def _compute_bands_and_gates(self, x: torch.Tensor) -> tuple[torch.Tensor, list[torch.Tensor], torch.Tensor]:
        h1 = self.layer1(self.stem(x))
        f2 = self.layer2(h1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        bands = self._band_vectors(f2) + self._band_vectors(f3)
        stats = [b.abs().mean(dim=1) for b in bands] + [b.pow(2).mean(dim=1).add(1e-6).sqrt() for b in bands]
        w = torch.softmax(self.gate(torch.stack(stats, dim=1)), dim=-1)  # (B, 8)
        return f4, bands, w

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f4, bands, w = self._compute_bands_and_gates(x)
        self._last_gates = w
        scaled = torch.cat([b * (w[:, i : i + 1] * self.n_components) for i, b in enumerate(bands)], dim=1)
        delta = self.out_proj(self.proj(scaled))
        backbone_vec = self.pool(f4).flatten(1)
        return self.head(backbone_vec + delta)

    def gate_entropy_loss(self) -> torch.Tensor:
        if self._last_gates is None:
            return torch.zeros((), device=next(self.parameters()).device)
        p = self._last_gates.clamp_min(1e-8)
        h = -(p * p.log()).sum(-1).mean()
        return (math.log(self.n_components) - h) / math.log(self.n_components)

    def gate_weights(self, x: torch.Tensor) -> torch.Tensor:
        was_training = self.training
        self.eval()
        with torch.no_grad():
            _f4, _bands, w = self._compute_bands_and_gates(x)
        if was_training:
            self.train()
        return w


# ---------------------------------------------------------------- CMP-ResNet18

_CMP_AGGREGATORS: dict[str, Callable[[torch.Tensor, torch.Tensor, int], torch.Tensor]] = {
    "hardmax": lambda sim, kappa, _k: sim.max(dim=2).values * kappa,
    "lse": lambda sim, kappa, k: torch.logsumexp(sim * kappa, dim=2) - math.log(k),
}


class CMPResNet18(nn.Module):
    """Confusion-guided Multi-Prototype ResNet18.

    - Backbone ResNet18 pretrained → GAP → embed Linear → L2 normalize.
    - K prototype/lớp (L2 normalize) → sim (B, C, K) → aggregation strategy (`lse` | `hardmax`).
    - Confusion margin (có hướng, chỉ trong loss): target=a → logit_b −= m·w[a,b].
    - Diversity reg (tùy chọn): mean abs-cos giữa prototype cùng lớp.
    """

    def __init__(self, num_classes: int = 45, pretrained: bool = True, n_proto: int = 1,
                 embed_dim: int = 512, confusion: bool = False, conf_margin: float = 0.3,
                 diversity: bool = False, agg: str = "lse", pair: bool = False,
                 pair_margin: float = 0.1, scale_mode: str = "linear"):
        super().__init__()
        import torchvision

        bb = torchvision.models.resnet18(
            weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        )
        self.backbone = nn.Sequential(*list(bb.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.embed = nn.Linear(512, embed_dim)
        self.C, self.K = num_classes, n_proto
        self.prototypes = nn.Parameter(torch.randn(num_classes * n_proto, embed_dim) * 0.02)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(10.0)))
        self.confusion = confusion
        self.conf_margin = conf_margin
        self.diversity = diversity
        self.cmp_agg = agg
        self.cmp_pair = pair
        self.pair_margin = pair_margin
        self.scale_mode = scale_mode
        self.register_buffer("conf_w", torch.zeros(num_classes, num_classes))

    def set_confusion(self, w: np.ndarray) -> None:
        assert w.shape == (self.C, self.C) and float(w.min()) >= 0.0 and float(w.max()) <= 1.0
        self.conf_w.copy_(torch.from_numpy(w.astype(np.float32)))

    def embeddings(self, x: torch.Tensor) -> torch.Tensor:
        z = self.pool(self.backbone(x)).flatten(1)
        return F.normalize(self.embed(z), dim=-1)

    def current_kappa(self) -> torch.Tensor:
        if self.scale_mode == "exp":
            return self.logit_scale.exp().clamp(max=64.0)
        return self.logit_scale.clamp(max=math.log(100.0))

    def _cosine_sim(self, e: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        p = F.normalize(self.prototypes, dim=-1)
        sim = (e @ p.T).view(-1, self.C, self.K)
        return sim, self.current_kappa()

    def _off_diag_conf_weights(self, labels: torch.Tensor) -> torch.Tensor:
        labels = labels.to(self.conf_w.device)
        return self.conf_w[labels] * (1.0 - F.one_hot(labels, self.C).float())

    def logits_from_emb(self, e: torch.Tensor, apply_margin: bool = False,
                        labels: torch.Tensor | None = None) -> torch.Tensor:
        sim, kappa = self._cosine_sim(e)
        agg_fn = _CMP_AGGREGATORS.get(self.cmp_agg, _CMP_AGGREGATORS["lse"])
        logits = agg_fn(sim, kappa, self.K)
        if apply_margin and self.confusion and labels is not None:
            logits = logits - self.conf_margin * self._off_diag_conf_weights(labels)
        return logits

    def forward(self, x: torch.Tensor, labels: torch.Tensor | None = None) -> torch.Tensor:
        return self.logits_from_emb(self.embeddings(x), apply_margin=True, labels=labels)

    def pair_loss(self, e: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        """CMP-PG: pair loss có hướng rõ ràng trên confusion graph train-only."""
        sim, kappa = self._cosine_sim(e)
        s = sim.max(dim=2).values * kappa
        w = self._off_diag_conf_weights(labels)
        z = w.sum()
        if z <= 0:
            return e.new_zeros(())
        labels_dev = labels.to(s.device)
        pair_terms = F.softplus(self.pair_margin + s - s.gather(1, labels_dev.unsqueeze(1)))
        return (w * pair_terms).sum() / z

    def diversity_loss(self) -> torch.Tensor:
        p = F.normalize(self.prototypes, dim=-1).view(self.C, self.K, -1)
        if self.K < 2:
            return torch.zeros((), device=p.device)
        cos = torch.bmm(p, p.transpose(1, 2))
        eye = torch.eye(self.K, device=p.device).unsqueeze(0)
        return ((cos - eye).abs().sum(dim=(1, 2)) / (self.K * (self.K - 1))).mean()

    def prototype_usage(self, loader: DataLoader, device: str, max_batches: int | None = None) -> dict:
        """Thống kê usage/khoảng cách prototype (phát hiện collapse)."""
        self.eval()
        assign = torch.zeros(self.C, self.K, dtype=torch.long)
        n_seen = 0
        with torch.no_grad():
            for bi, (x, y) in enumerate(loader):
                if max_batches and bi >= max_batches:
                    break
                e = self.embeddings(x.to(device))
                p = F.normalize(self.prototypes, dim=-1).view(self.C, self.K, -1)
                pc = p[y.to(p.device)]
                sim = torch.einsum("bd,bkd->bk", e, pc)
                for cls, proto in zip(y.tolist(), sim.argmax(1).cpu().tolist(), strict=True):
                    assign[cls, proto] += 1
                n_seen += len(y)
        inter = []
        with torch.no_grad():
            p = F.normalize(self.prototypes, dim=-1).view(self.C, self.K, -1).cpu()
            if self.K > 1:
                cos = torch.bmm(p, p.transpose(1, 2))
                eye = torch.eye(self.K).bool().unsqueeze(0)
                inter = cos.masked_fill(eye, 0).amax(dim=(1, 2)).tolist()
        a_np = assign.numpy()
        minority = float(np.mean([min(r) / max(sum(r), 1) for r in a_np if sum(r) > 0])) if len(a_np) else 0.0
        return {"n_seen": n_seen, "assign_per_class_proto": assign.tolist(),
                "unused_proto_classes": int((assign.sum(1) == 0).sum()),
                "minority_ratio": minority,
                "max_inter_proto_cos_mean": float(np.mean(inter)) if inter else None}


def init_prototypes_kmeans(model: CMPResNet18, manifest: pd.DataFrame, root: str, device: str,
                           batch_size: int = 256, num_workers: int = 4, seed: int = 0,
                           log_every: int = 20) -> dict:
    """Khởi tạo prototype bằng k-means trên embedding ImageNet của ảnh TRAIN (eval transform)."""
    from sklearn.cluster import KMeans

    model.to(device).eval()
    dl = DataLoader(ResiscDataset(manifest, root, train=False, input_size=256), batch_size=batch_size,
                    shuffle=False, num_workers=num_workers)
    embs, ys = [], []
    t0 = time.time()
    with torch.no_grad():
        for bi, (x, y) in enumerate(dl):
            embs.append(model.embeddings(x.to(device)).cpu())
            ys.append(y)
            if (bi + 1) % log_every == 0:
                print(f"[kmeans-init] batch {bi + 1}/{len(dl)} | {time.time() - t0:.0f}s", flush=True)
    E = torch.cat(embs).numpy()
    Y = torch.cat(ys).numpy()
    C, K = model.C, model.K
    protos = np.zeros((C * K, E.shape[1]), dtype=np.float32)
    inertias = []
    for c in range(C):
        Ec = E[Y == c]
        if len(Ec) >= K:
            km = KMeans(n_clusters=K, n_init=4, random_state=seed).fit(Ec)
            protos[c * K:(c + 1) * K] = km.cluster_centers_
            inertias.append(float(km.inertia_))
        else:
            protos[c * K:(c + 1) * K] = model.prototypes.data[c * K:(c + 1) * K].cpu().numpy()
    model.prototypes.data = torch.from_numpy(protos).to(model.prototypes.device)
    info = {"n_embed": len(E), "mean_inertia": float(np.mean(inertias)) if inertias else None,
            "elapsed_sec": round(time.time() - t0, 1)}
    print(f"[kmeans-init] DONE {info}", flush=True)
    return info


# ---------------------------------------------------------------- data staging


def stage_dataset(manifest: pd.DataFrame, src_root: str, dst_root: str, num_threads: int = 16,
                  log_every: int = 2000) -> str:
    """Copy ảnh từ mount FUSE (/kaggle/input) sang disk local MỘT lần song song theo thread."""
    import hashlib
    import shutil
    from concurrent.futures import ThreadPoolExecutor

    os.makedirs(dst_root, exist_ok=True)
    ids = manifest.image_id.tolist()

    def copy_one(rel: str) -> tuple[str, str]:
        dst = os.path.join(dst_root, rel)
        d = os.path.dirname(dst)
        if not os.path.isdir(d):
            os.makedirs(d, exist_ok=True)
        if not os.path.exists(dst):
            shutil.copy2(os.path.join(src_root, rel), dst)
        with open(dst, "rb") as f:
            return rel, hashlib.md5(f.read()).hexdigest()

    t0 = time.time()
    hashes = {}
    with ThreadPoolExecutor(max_workers=num_threads) as ex:
        for k, (rel, h) in enumerate(ex.map(copy_one, ids), start=1):
            hashes[rel] = h
            if k % log_every == 0 or k == len(ids):
                el = time.time() - t0
                print(f"[stage] {k}/{len(ids)} ({100 * k / len(ids):.0f}%) | {el:.0f}s | "
                      f"{k / el:.0f}/s | ETA {el / k * (len(ids) - k):.0f}s", flush=True)
    staged = sum(len(files) for _, _, files in os.walk(dst_root))
    assert staged == len(ids), f"staged {staged} != {len(ids)}"
    sample = ids[:: max(len(ids) // 100, 1)][:100]
    for rel in sample:
        with open(os.path.join(src_root, rel), "rb") as f:
            src_h = hashlib.md5(f.read()).hexdigest()
        assert hashes[rel] == src_h, f"MD5 lệch sau stage: {rel}"
    print(f"[stage] VERIFY PASS: {staged} file, {len(sample)} MD5 sample khớp nguồn | "
          f"tổng {time.time() - t0:.0f}s")
    return dst_root


# ---------------------------------------------------------------- confusion graph (train-only)


def build_confusion_graph(train_manifest: pd.DataFrame, root: str, graph_frac: float = 0.10,
                          seed: int = 0, probe_epochs: int = 6, device: str = "cuda",
                          min_count: int = 3, num_classes: int = 45,
                          num_workers: int = 4) -> tuple[np.ndarray, dict]:
    """Probe model train trên (1−graph_frac) của TRAIN, dự đoán trên phần graph còn lại (cùng train)."""
    rng = np.random.default_rng(seed)
    fit_df, graph_df = [], []
    for _cls, gdf in train_manifest.groupby("class_name"):
        idx = rng.permutation(len(gdf))
        n_graph = max(1, round(len(gdf) * graph_frac))
        graph_df.append(gdf.iloc[idx[:n_graph]])
        fit_df.append(gdf.iloc[idx[n_graph:]])
    fit_df, graph_df = pd.concat(fit_df), pd.concat(graph_df)

    pc = TrainConfig(model_key="resnet18", num_classes=num_classes, epochs=probe_epochs, seed=seed,
                     num_workers=num_workers)
    probe = build_model(pc)
    dl_fit = DataLoader(ResiscDataset(fit_df, root, True, pc.input_size), batch_size=pc.batch_size, shuffle=True,
                        num_workers=num_workers)
    dl_graph = DataLoader(ResiscDataset(graph_df, root, False, pc.input_size), batch_size=pc.batch_size, shuffle=False,
                          num_workers=num_workers)
    probe = train_model(probe, pc, dl_fit, None, device, verbose=False)[0]

    n = np.zeros((num_classes, num_classes), dtype=np.float64)
    probe.eval()
    with torch.no_grad():
        for x, y in dl_graph:
            pred = probe(x.to(device)).argmax(1).cpu().numpy()
            for a, b in zip(y.numpy(), pred, strict=True):
                n[a, b] += 1
    off = n.copy()
    np.fill_diagonal(off, 0.0)
    row_max = off.max(axis=1, keepdims=True)
    w = np.where((off >= min_count) & (row_max > 0), off / np.maximum(row_max, 1e-9), 0.0)
    info = {"probe_epochs": probe_epochs, "n_fit": len(fit_df), "n_graph": len(graph_df),
            "n_pairs_active": int((w > 0).sum()), "max_n": int(off.max()),
            "confusion_rate_graph": float(off.sum() / max(len(graph_df), 1))}
    return w.astype(np.float32), info


# ---------------------------------------------------------------- train / eval strategies


def _param_groups(model: nn.Module, cfg: TrainConfig) -> list[dict]:
    """Backbone lr thấp hơn head lr — pretrained conv là backbone."""
    head_keywords = ("head", "fc", "classifier", "embed", "prototypes", "logit_scale",
                     "proj", "gate", "out_proj", "fixed_logits")
    head, backbone = [], []
    for name, p in model.named_parameters():
        (head if any(k in name for k in head_keywords) else backbone).append(p)
    return [{"params": backbone, "lr": cfg.lr_backbone}, {"params": head, "lr": cfg.lr_head}]


def kd_loss(student_logits: torch.Tensor, teacher_logits: torch.Tensor, T: float) -> torch.Tensor:
    """Knowledge distillation chuẩn (Hinton et al.): T² · KL(softmax(t/T) ‖ softmax(s/T))."""
    return F.kl_div(F.log_softmax(student_logits.float() / T, dim=1),
                    F.softmax(teacher_logits.float() / T, dim=1),
                    reduction="batchmean") * (T * T)


class KDDistiller:
    """Strategy bọc teacher frozen cho Knowledge Distillation cùng batch (same-view)."""

    def __init__(self, teacher: nn.Module, device: str, alpha: float, temperature: float):
        self.teacher = teacher.to(device).eval()
        for p in self.teacher.parameters():
            p.requires_grad_(False)
        self.alpha = alpha
        self.temperature = temperature

    def blend_loss(self, base_loss: torch.Tensor, student_logits: torch.Tensor,
                   x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        with torch.no_grad():
            t_logits = self.teacher(x)
        kd_l = kd_loss(student_logits, t_logits, self.temperature)
        total = (1.0 - self.alpha) * base_loss + self.alpha * kd_l
        return total, kd_l


class EarlyStopper:
    """Theo dõi val macro-F1, giữ best state_dict, lưu checkpoint và dừng sớm khi hết patience."""

    def __init__(self, patience: int, state_out: str | None = None):
        self.patience = patience
        self.state_out = state_out
        self.best_f1 = -1.0
        self.best_epoch = -1
        self.best_state: dict | None = None
        self.bad_epochs = 0

    def update(self, val_f1: float, epoch: int, model: nn.Module, cfg: TrainConfig) -> bool:
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.best_epoch = epoch
            self.best_state = copy.deepcopy(model.state_dict())
            self.bad_epochs = 0
            if self.state_out:
                torch.save({"state_dict": self.best_state, "config": cfg.to_dict(), "epoch": epoch}, self.state_out)
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience


def _compute_step_loss(
    model: nn.Module,
    crit: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    cfg: TrainConfig,
    pair_w: float,
    ent_w: float,
    aux: dict[str, float],
    distiller: KDDistiller | None,
) -> torch.Tensor:
    """Tính loss 1 step (base CE + aux losses theo model capability + KD nếu bật)."""
    logits = model(x, y) if isinstance(model, CMPResNet18) else model(x)
    loss = crit(logits, y)
    n = len(y)
    if isinstance(model, CMPResNet18) and model.diversity:
        div_l = model.diversity_loss()
        loss = loss + cfg.cmp_div_weight * div_l
        aux["div"] += float(div_l.detach()) * n
    if isinstance(model, CMPResNet18) and model.cmp_pair and model.confusion:
        pair_l = model.pair_loss(model.embeddings(x), y)
        loss = loss + pair_w * pair_l
        aux["pair"] += float(pair_l.detach()) * n
    if ent_w > 0 and hasattr(model, "gate_entropy_loss"):
        ent_l = model.gate_entropy_loss()
        loss = loss + ent_w * ent_l
        aux["gate_ent"] += float(ent_l.detach()) * n
    if distiller is not None:
        loss, kd_l = distiller.blend_loss(loss, logits, x)
        aux["kd"] += float(kd_l.detach()) * n
    return loss


def _make_cosine_warmup_scheduler(opt: torch.optim.Optimizer, epochs: int, steps_per_epoch: int,
                                  warmup_frac: float) -> torch.optim.lr_scheduler.LambdaLR:
    total_steps = epochs * max(steps_per_epoch, 1)
    warmup = max(1, int(warmup_frac * total_steps))

    def lr_lambda(step: int) -> float:
        if step < warmup:
            return step / warmup
        p = (step - warmup) / max(total_steps - warmup, 1)
        return 0.5 * (1 + math.cos(math.pi * p))

    return torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)


def train_model(model: nn.Module, cfg: TrainConfig, dl_train: DataLoader, dl_val: DataLoader | None,
                device: str, verbose: bool = True, state_out: str | None = None,
                teacher: nn.Module | None = None) -> tuple[nn.Module, dict]:
    """Train với AMP + cosine + warmup + early stop (val macro-F1). Trả về (best_model, history)."""
    model.to(device)
    distiller = KDDistiller(teacher, device, cfg.kd_alpha, cfg.kd_T) if teacher is not None else None
    opt = torch.optim.AdamW(_param_groups(model, cfg), weight_decay=cfg.weight_decay)
    sched = _make_cosine_warmup_scheduler(opt, cfg.epochs, len(dl_train), cfg.warmup_frac)
    use_cuda_amp = cfg.amp and device.startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda_amp)
    crit = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    stopper = EarlyStopper(cfg.early_stop_patience, state_out=state_out)
    history: list[dict] = []
    t_train0 = time.time()

    for ep in range(cfg.epochs):
        model.train()
        ep_loss, ep_n, t0 = 0.0, 0, time.time()
        aux = {"gate_ent": 0.0, "div": 0.0, "pair": 0.0, "kd": 0.0}
        n_batches = len(dl_train)
        ent_w = cfg.asf_gate_ent
        if cfg.asf_gate_ent_anneal > 0:
            ent_w = cfg.asf_gate_ent * max(0.0, 1.0 - ep / cfg.asf_gate_ent_anneal)
        pair_w = cfg.cmp_pair_weight * min(1.0, (ep + 1) / max(cfg.cmp_pair_warmup, 1))

        for bi, (x, y) in enumerate(dl_train, start=1):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_cuda_amp):
                loss = _compute_step_loss(model, crit, x, y, cfg, pair_w, ent_w, aux, distiller)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            ep_loss += loss.item() * len(y)
            ep_n += len(y)
            if bi % 50 == 0 or bi == n_batches:
                el = time.time() - t0
                print(f"[train] ep {ep} batch {bi}/{n_batches} | {el:.0f}s | "
                      f"loss={(ep_loss / max(ep_n, 1)):.3f} | ETA {el / bi * (n_batches - bi):.0f}s", flush=True)

        entry = {"epoch": ep, "train_loss": ep_loss / max(ep_n, 1), "epoch_sec": round(time.time() - t0, 2),
                 "lr": sched.get_last_lr()[0],
                 "aux": {k: round(v / max(ep_n, 1), 6) for k, v in aux.items()},
                 "ent_weight": round(ent_w, 4), "pair_weight": round(pair_w, 4)}
        should_stop = False
        if dl_val is not None:
            m = evaluate(model, dl_val, device)
            entry.update({"val_acc": m["acc"], "val_macro_f1": m["macro_f1"]})
            should_stop = stopper.update(m["macro_f1"], ep, model, cfg)
        history.append(entry)
        if verbose:
            print(json.dumps(entry))
        if should_stop:
            if verbose:
                print(f"early stop @ epoch {ep} (best {stopper.best_epoch})")
            break

    train_time = time.time() - t_train0
    if stopper.best_state is not None:
        model.load_state_dict(stopper.best_state)
    hist_summary = {"train_time_sec": round(train_time, 1), "executed_epochs": len(history),
                    "best_epoch": stopper.best_epoch,
                    "best_val_macro_f1": None if stopper.best_f1 < 0 else stopper.best_f1,
                    "history": history}
    return model, hist_summary


@torch.inference_mode()
def evaluate(model: nn.Module, loader: DataLoader, device: str) -> dict:
    """Full metrics: acc + macro P/R/F1 + per-class + confusion matrix."""
    from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

    model.to(device).eval()
    ys, ps = [], []
    for x, y in loader:
        with torch.amp.autocast("cuda", enabled=device.startswith("cuda")):
            logits = model(x.to(device))
        ps.append(logits.argmax(1).cpu())
        ys.append(y)
    y_true, y_pred = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    pp, rr, ff, sup = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    return {"acc": float((y_true == y_pred).mean()), "macro_precision": float(p), "macro_recall": float(r),
            "macro_f1": float(f1),
            "per_class": {"precision": pp.tolist(), "recall": rr.tolist(), "f1": ff.tolist(), "support": sup.tolist()},
            "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(), "n": len(y_true)}


@torch.inference_mode()
def measure_latency(model: nn.Module, device: str, input_size: int = 256, n_warmup: int = 10,
                    n_iter: int = 50) -> dict:
    """Latency model-only, batch 1, median/p95 (ms). model.eval() + cuda sync + warmup."""
    model.to(device).eval()
    x = torch.randn(1, 3, input_size, input_size, device=device)
    for _ in range(n_warmup):
        model(x)
    if device.startswith("cuda"):
        torch.cuda.synchronize()
    times = []
    for _ in range(n_iter):
        if device.startswith("cuda"):
            t0 = torch.cuda.Event(enable_timing=True)
            t1 = torch.cuda.Event(enable_timing=True)
            t0.record()
            model(x)
            t1.record()
            torch.cuda.synchronize()
            times.append(t0.elapsed_time(t1))
        else:
            t0 = time.perf_counter()
            model(x)
            times.append((time.perf_counter() - t0) * 1000)
    return {"median_ms": float(np.median(times)), "p95_ms": float(np.percentile(times, 95)),
            "n_iter": n_iter, "batch": 1}


In [ ]:
# ---- STAGE + teacher ----
if DATA_ROOT.startswith("/kaggle/"):
    DATA_ROOT = stage_dataset(manifest, DATA_ROOT, "/tmp/resisc45_staged/NWPU-RESISC45", num_threads=NUM_WORKERS * 4)
else:
    print("local run — bỏ qua stage")

teacher_rel = f"ckpt_convnextt_s{TEACHER_SEED}.pt"
teacher_path = find_single([f"/kaggle/input/**/{teacher_rel}",
                            f"../../outputs/resisc45-final-baselines-b0-b4-3-seeds/{teacher_rel}"],
                           f"teacher {teacher_rel}")
with open(teacher_path, "rb") as tf:
    teacher_sha = hashlib.sha256(tf.read()).hexdigest()
if not SMOKE:
    assert teacher_sha.startswith("9627ca43"), "FAIL-FAST: teacher checkpoint lệch"
teacher = build_model(TrainConfig(num_classes=45, model_key="convnextt"), pretrained=False).to(device)
sd = torch.load(teacher_path, map_location=device, weights_only=False)
teacher.load_state_dict(sd["state_dict"])
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)
print(f"teacher verified: seed {TEACHER_SEED}, sha {teacher_sha[:16]}…, epoch {sd['epoch']}")

In [ ]:
# ---- Screening loop ----
results = []
REPORT = "h1a_screen_report.json"


def save_report() -> None:
    with open(REPORT, "w") as f:
        json.dump({"run_id": RUN_ID, "smoke": SMOKE, "device": device, "seed": SEED,
                   "epochs": EPOCHS, "kd": {"alpha": KD_ALPHA, "T": KD_T},
                   "teacher": {"seed": TEACHER_SEED, "sha256": teacher_sha},
                   "manifest_sha256": audit["split_sha256"],
                   "ref_resnet18_s0": REF_S0, "results": results}, f, indent=1, ensure_ascii=False)


train_manifest = manifest[manifest.split == "train"].reset_index(drop=True)
for name, overrides in EXPERIMENTS:
    ck = f"ckpt_{name}.pt"
    set_seed(SEED)
    cfg = TrainConfig(num_classes=45, seed=SEED, epochs=EPOCHS, num_workers=NUM_WORKERS,
                      amp=device.startswith("cuda"), **overrides)
    student = build_model(cfg, pretrained=not SMOKE).to(device)
    kmeans_info = None
    if cfg.model_key == "cmp" and cfg.cmp_kmeans_init:
        kmeans_info = init_prototypes_kmeans(student, train_manifest, DATA_ROOT, device, seed=SEED,
                                             num_workers=NUM_WORKERS)
    params = count_params(student)
    dl_tr, dl_va = make_loaders(manifest, DATA_ROOT, cfg)
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    print(f"\n=== {name} | params {params['total']} ===", flush=True)
    use_teacher = teacher if cfg.kd_alpha > 0 else None
    student, hist = train_model(student, cfg, dl_tr, dl_va, device,
                                state_out=ck, teacher=use_teacher)
    peak = torch.cuda.max_memory_allocated() / 2**30 if device == "cuda" else None
    m = evaluate(student, dl_va, device)
    lat = measure_latency(student, device, input_size=cfg.input_size,
                          n_warmup=2 if SMOKE else 10, n_iter=5 if SMOKE else 50)
    usage, gate_summary, kappa_val = None, None, None
    if cfg.model_key == "cmp":
        usage = student.prototype_usage(dl_va, device, max_batches=2 if SMOKE else None)
        kappa_val = round(float(student.current_kappa().item()), 4)
    if cfg.model_key == "asf_lite":
        xb_s, _ = next(iter(dl_va))
        gw = student.gate_weights(xb_s[:16].to(device)).cpu()
        gate_summary = {"gate_mean": [round(v, 4) for v in gw.mean(0).tolist()],
                        "gate_std": [round(v, 5) for v in gw.std(0).tolist()],
                        "max_mean": round(float(gw.mean(0).max()), 4)}
    ref = REF_S0["kd" if cfg.kd_alpha > 0 else "ce"]
    entry = {"name": name, "config": cfg.to_dict(), "params": params,
             "best_val_acc": round(m["acc"], 4), "best_val_macro_f1": round(m["macro_f1"], 4),
             "delta_vs_resnet18_s0": round(m["macro_f1"] - ref, 4),
             "latency_ms": lat, "kappa": kappa_val,
             "history": hist["history"],
             "train_time_sec": hist["train_time_sec"], "best_epoch": hist["best_epoch"],
             "executed_epochs": hist["executed_epochs"], "peak_mem_gib": peak,
             "prototype_usage": usage, "gate_summary": gate_summary,
             "checkpoint": ck}
    results.append(entry)
    save_report()
    print(json.dumps({k: v for k, v in entry.items() if k not in ("history", "prototype_usage")},
                     ensure_ascii=False), flush=True)

In [ ]:
# ---- Summary & checks ----
with open(REPORT) as f:
    rep = json.load(f)
assert len(rep["results"]) == len(EXPERIMENTS)
for r in rep["results"]:
    print(f"{r['name']:<14} | valF1 {r['best_val_macro_f1']:.4f} (ΔRN18 {r['delta_vs_resnet18_s0']:+.4f}) "
          f"| lat {r['latency_ms']['median_ms']:.2f}ms | kappa={r['kappa']} | params={r['params']['total']/1e6:.2f}M")
print("H1A SCREEN DONE")